
# Smart MCQ Solver Challenge

Models:

1. BiLSTM (from scratch) - 5 epochs
2. DistilBERT - 3 epochs
3. RoBERTa - 3 epochs

Key fixes:

- `eval_strategy="epoch"` for latest Transformers versions
- `fp16=False` for Kaggle GPU compatibility
- `wandb.finish()` is called **after predictions**
- Prediction happens while the WandB run is still active
- Shared training function for DistilBERT and RoBERTa


In [1]:

import random
import numpy as np
import pandas as pd
import torch
import transformers
import wandb

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA: 12.8
Transformers: 5.0.0
GPU: Tesla T4


In [2]:

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

label2id = {'A':0,'B':1,'C':2,'D':3,'E':4}
id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=SEED
)


In [3]:

WANDB_API_KEY = "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"

PROJECT = "23f2000003-t22026"
ENTITY = "23f2000003-t22026"

wandb.login(key=WANDB_API_KEY)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:

def map3_score(y_true, preds):

    scores = []

    for y, pred in zip(y_true, preds):

        score = 0

        for rank, p in enumerate(pred[:3], start=1):

            if p == y:
                score = 1 / rank
                break

        scores.append(score)

    return np.mean(scores)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    top3 = np.argsort(-logits, axis=1)[:, :3]

    return {
        "accuracy": accuracy_score(labels, top3[:, 0]),
        "weighted_f1": f1_score(labels, top3[:, 0], average="weighted"),
        "map3": map3_score(labels, top3)
    }


In [5]:

def build_dataset(model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(example):

        result = tokenizer(
            [example["prompt"]] * 5,
            [example["A"], example["B"], example["C"], example["D"], example["E"]],
            truncation=True,
            padding="max_length",
            max_length=256
        )

        result["labels"] = example["label"]

        return result

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
    valid_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))
    test_ds = Dataset.from_pandas(test.reset_index(drop=True))

    train_ds = train_ds.map(preprocess)
    valid_ds = valid_ds.map(preprocess)

    def preprocess_test(example):

        return tokenizer(
            [example["prompt"]] * 5,
            [example["A"], example["B"], example["C"], example["D"], example["E"]],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    test_ds = test_ds.map(preprocess_test)

    keep_train = ["input_ids", "attention_mask", "labels"]
    keep_test = ["input_ids", "attention_mask"]

    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_train])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep_train])
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_test])

    train_ds.set_format("torch")
    valid_ds.set_format("torch")
    test_ds.set_format("torch")

    return tokenizer, train_ds, valid_ds, test_ds


In [6]:

def train_transformer(model_name, run_name, epochs):

    wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=run_name,
        reinit=True
    )

    tokenizer, train_ds, valid_ds, test_ds = build_dataset(model_name)

    model = AutoModelForMultipleChoice.from_pretrained(model_name)

    args = TrainingArguments(
        output_dir=f"./{run_name}",
        run_name=run_name,

        num_train_epochs=epochs,
        learning_rate=2e-5,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="map3",
        greater_is_better=True,

        report_to="wandb",

        fp16=False,
        bf16=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    predictions = trainer.predict(test_ds)

    top3 = np.argsort(-predictions.predictions, axis=1)[:, :3]

    submission = pd.DataFrame({
        "id": test["id"],
        "Prediction": [
            " ".join(id2label[x] for x in row)
            for row in top3
        ]
    })

    submission.to_csv(f"{run_name}_submission.csv", index=False)

    wandb.finish()

    return submission


## Model 1 - DistilBERT

In [7]:

distilbert_submission = train_transformer(
    model_name="distilbert-base-uncased",
    run_name="distilbert",
    epochs=3
)

distilbert_submission.head()


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_141642-h06o4uia
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert
wandb: ⭐️ View project at https://wandb.ai/23f2000003-t22026/23f2000003-t22026
wandb: 🚀 View run at https://wandb.ai/23f2000003-t22026/23f2000003-t22026/runs/h06o4uia


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


wandb: updating run metadata
wandb: uploading history steps 5-5, summary
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁██
wandb:               eval/loss █▃▁
wandb:               eval/map3 ▁██
wandb:            eval/runtime █▄▁
wandb: eval/samples_per_second ▁▅█
wandb:   eval/steps_per_second ▁▅█
wandb:        eval/weighted_f1 ▁██
wandb:            test/runtime ▁
wandb: test/samples_per_second ▁
wandb:   test/steps_per_second ▁
wandb:                      +5 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.9325
wandb:               eval/loss 0.72081
wandb:               eval/map3 0.95667
wandb:            eval/runtime 8.5365
wandb: eval/samples_per_second 46.858
wandb:   eval/steps_per_second 5.857
wandb:        eval/weighted_f1 0.9327
wandb:            test/runtime 10.6068
wandb: test/samples_per_second 47.139
wandb:   test/steps_per_second 5.94
wandb:                     +10 ...
wandb: 
wandb: 🚀 View run distilbert at: https://wandb.ai/23f2000003-t22026/2

,id,Prediction
0,1,A B E
1,2,E B A
2,3,B D E
3,4,E C D
4,5,C A D


## Model 2 - RoBERTa

In [8]:

roberta_submission = train_transformer(
    model_name="roberta-base",
    run_name="roberta",
    epochs=3
)

roberta_submission.head()


wandb: setting up run 5i7c1gm4
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_142230-5i7c1gm4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run roberta
wandb: ⭐️ View project at https://wandb.ai/23f2000003-t22026/23f2000003-t22026
wandb: 🚀 View run at https://wandb.ai/23f2000003-t22026/23f2000003-t22026/runs/5i7c1gm4


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

wandb: updating run metadata
wandb: uploading history steps 5-5, summary
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁██
wandb:               eval/loss █▂▁
wandb:               eval/map3 ▁█▇
wandb:            eval/runtime █▂▁
wandb: eval/samples_per_second ▁▇█
wandb:   eval/steps_per_second ▁▇█
wandb:        eval/weighted_f1 ▁██
wandb:            test/runtime ▁
wandb: test/samples_per_second ▁
wandb:   test/steps_per_second ▁
wandb:                      +5 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.9575
wandb:               eval/loss 0.39675
wandb:               eval/map3 0.97042
wandb:            eval/runtime 15.9482
wandb: eval/samples_per_second 25.081
wandb:   eval/steps_per_second 3.135
wandb:        eval/weighted_f1 0.95708
wandb:            test/runtime 19.9126
wandb: test/samples_per_second 25.11
wandb:   test/steps_per_second 3.164
wandb:                     +10 ...
wandb: 
wandb: 🚀 View run roberta at: https://wandb.ai/23f2000003-t22026/23

,id,Prediction
0,1,A D B
1,2,B D A
2,3,B E D
3,4,C E A
4,5,C A D



## Model 3 - BiLSTM



